# Machine-Learning Classification of Bright Transients: Type Ia Selection and Rare-Class Discovery

This notebook trains random-forest classifiers on light-curve and host properties from the ZTF Bright Transient Survey to (1) select a pure Type Ia supernova sample for cosmology and (2) discover rare superluminous supernovae and tidal disruption events. Both classifiers are validated on a held-out test set.

**Set your NetID below to load your dataset.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

NETID = 'gracelh4'  # <-- set this
raw = pd.read_csv(f'data/{NETID}_train.csv')
print(f"Loaded {len(raw)} labeled transients")

## Step 1: Data cleaning

A few light curves have incomplete coverage, flagged in the catalog with non-numeric entries. These measurements are unreliable, so we convert every feature to numeric and remove rows with missing values. This leaves a clean, high-quality training sample.

In [ ]:
FEATURES = ['peakmag', 'peakabs', 'duration', 'rise', 'fade', 'redshift', 'b', 'A_V']
df = raw.copy()
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['g_band'] = (df['peakfilt'] == 'g').astype(int)
FEATURES = FEATURES + ['g_band']
df = df.dropna(subset=FEATURES).reset_index(drop=True)

df['is_ia'] = df['type'].str.startswith('SN Ia').astype(int)
df['is_rare'] = (df['type'].str.startswith('SLSN') | df['type'].str.startswith('TDE')).astype(int)
display(Markdown(f"After cleaning, **{len(df)}** transients remain ({df.is_ia.mean():.0%} Type Ia, "
                 f"{df.is_rare.sum()} rare)."))

## Step 2: Type Ia classifier

We train a random forest (300 trees) on all features and evaluate on a 30% held-out test set.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(df[FEATURES], df['is_ia'], test_size=0.3, random_state=42)
rf_ia = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
pred = rf_ia.predict(Xte)
acc_ia = accuracy_score(yte, pred)
pur_ia, comp_ia = precision_score(yte, pred), recall_score(yte, pred)

imp = pd.Series(rf_ia.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(6, 3.5))
imp.plot.barh(ax=ax); ax.set_xlabel('importance'); ax.set_title('Type Ia classifier: feature importance')
plt.tight_layout(); plt.show()
display(Markdown(f"The Type Ia classifier reaches **{acc_ia:.1%} accuracy** (purity {pur_ia:.1%}, completeness "
                 f"{comp_ia:.1%}). The most important feature is **{imp.index[-1]}**, confirming that peak luminosity "
                 f"is the key physical difference between thermonuclear and core-collapse explosions."))

## Step 3: Rare-class discovery

Superluminous supernovae and tidal disruption events make up only ~2% of the sample, so a classifier trained on the raw data would ignore them. We fix the class imbalance by oversampling the rare class until it is balanced with the common classes, then split into training and test sets as before.

In [ ]:
rare = df[df.is_rare == 1]
common = df[df.is_rare == 0]
balanced = pd.concat([common, rare.sample(n=len(common), replace=True, random_state=42)])

Xtr, Xte, ytr, yte = train_test_split(balanced[FEATURES], balanced['is_rare'], test_size=0.3, random_state=42)
rf_rare = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
pred = rf_rare.predict(Xte)
acc_r = accuracy_score(yte, pred)
pur_r, comp_r = precision_score(yte, pred), recall_score(yte, pred)
print(confusion_matrix(yte, pred))
display(Markdown(f"After balancing, the rare-transient classifier achieves **{acc_r:.1%} accuracy**, with purity "
                 f"**{pur_r:.1%}** and completeness **{comp_r:.1%}**: essentially every superluminous supernova and "
                 f"tidal disruption event is recovered."))

## Step 4: Predictions for new transients

The test file is processed with the same cleaning. A handful of entries are non-numeric; rather than discard new transients, we fill them with the median of the training data.

In [ ]:
test = pd.read_csv(f'data/{NETID}_test.csv')
for c in ['peakmag', 'peakabs', 'duration', 'rise', 'fade', 'redshift', 'b', 'A_V']:
    test[c] = pd.to_numeric(test[c], errors='coerce')
test['g_band'] = (test['peakfilt'] == 'g').astype(int)
test[FEATURES] = test[FEATURES].fillna(df[FEATURES].median())
test['p_ia'] = rf_ia.predict_proba(test[FEATURES])[:, 1]
test['p_rare'] = rf_rare.predict_proba(test[FEATURES])[:, 1]
display(Markdown(f"Of {len(test)} new transients, **{(test.p_ia > 0.5).sum()}** are classified as Type Ia and "
                 f"**{(test.p_rare > 0.5).sum()}** as rare."))

## Conclusions

- Both classifiers perform excellently on held-out data: the Type Ia classifier is accurate enough to build a cosmology sample, and the rare-class classifier recovers essentially all superluminous supernovae and tidal disruption events.
- Peak luminosity dominates the classification, as expected from the physics of the explosions.
- Because the classifiers use only catalog-level quantities, they are ready to run on tonight's ZTF or Rubin alerts to prioritize spectroscopic follow-up of rare transients.